# Step 1 — Corporate Finance: ML-Driven Project Funding Decisions
### Aurora Finance | e-PGD Capstone | IITB

---

**Business Problem:** Aurora Finance has 50 internal projects competing for limited capital.  
We use ML to decide which projects to fund — replacing gut-feel NPV rankings with data-driven decisions.

**Deliverables:**
1. Forecast project cash flows using Random Forest / XGBoost regression  
2. Predict project success probability using classification  
3. Rank projects by Expected Economic Value (EV) and risk-adjusted return  
4. SHAP plots for executive explainability  
5. Final ranked recommendation table + executive summary

---

In [1]:
!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn plotly scipy statsmodels imbalanced-learn ipykernel --quiet

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

# ── Try sklearn / XGBoost / SHAP; fall back to numpy if not installed ─────────
try:
    from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import (classification_report, roc_auc_score,
                                  mean_absolute_error, r2_score)
    SKLEARN = True
    print('sklearn loaded ')
except ImportError:
    SKLEARN = False
    print('sklearn not found — using numpy fallback')

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGB = True
    print('xgboost loaded ')
except ImportError:
    XGB = False
    print('xgboost not found — Random Forest will be used as primary model')

try:
    import shap
    SHAP = True
    print('shap loaded ')
except ImportError:
    SHAP = False
    print('shap not found — feature importance plot used as fallback')

sklearn loaded 
xgboost loaded 
shap loaded 


---
## Section 1 — Load Data & Financial Feature Engineering

We first compute the core financial metrics that every corporate finance decision rests on.
These become **features** for the ML models — they capture financial signal that raw cash flow numbers alone don't.

In [3]:
df = pd.read_csv('corporate_projects.csv')
print(f'Projects: {len(df)}  |  Features: {df.shape[1]}')
print(f'Success rate: {df["Success"].mean():.1%}')
df.head()

Projects: 50  |  Features: 10
Success rate: 52.0%


,Project_ID,Department,Investment_Cost,Expected_Cashflow_Year1,Expected_Cashflow_Year2,Expected_Cashflow_Year3,Historical_ROI,Market_Growth,Project_Risk,Success
0,1,Operations,184654,583556,2348816,1888756,0.08,0.05,Low,0
1,2,Marketing,2054354,1447527,541234,278576,0.15,-0.01,Low,1
2,3,Finance,2516182,1372905,1998260,2312990,0.19,0.02,High,1
3,4,Operations,3574675,1817640,189182,1434250,0.22,0.10,Medium,1
4,5,Operations,2752991,792139,1571101,111087,0.12,0.09,High,0


In [4]:
# ── 1A: Net Present Value ─────────────────────────────────────────────────────
# NPV = -Investment + CF1/(1+r) + CF2/(1+r)^2 + CF3/(1+r)^3
# Discount rate r = 10% (base-case WACC for Aurora)
r = 0.10

df['NPV'] = (
    -df['Investment_Cost']
    + df['Expected_Cashflow_Year1'] / (1 + r)**1
    + df['Expected_Cashflow_Year2'] / (1 + r)**2
    + df['Expected_Cashflow_Year3'] / (1 + r)**3
).round(0)

# ── 1B: Profitability Index ───────────────────────────────────────────────────
# PI = (NPV + Investment) / Investment
# PI > 1 means value-creating; useful for ranking when capital is constrained
df['PI'] = ((df['NPV'] + df['Investment_Cost']) / df['Investment_Cost']).round(3)

# ── 1C: Internal Rate of Return (Newton-Raphson) ──────────────────────────────
def compute_irr(inv, cf1, cf2, cf3):
    """Find discount rate where NPV = 0, using Newton-Raphson iteration."""
    cfs = [-inv, cf1, cf2, cf3]
    r   = 0.10  # initial guess
    for _ in range(500):
        f  = sum(c / (1+r)**t for t, c in enumerate(cfs))
        fp = sum(-t * c / (1+r)**(t+1) for t, c in enumerate(cfs))
        if abs(fp) < 1e-10 or r < -0.99 or r > 100:
            return np.nan
        r_new = r - f / fp
        if abs(r_new - r) < 1e-7:
            return round(r_new * 100, 2)
        r = r_new
    return np.nan

df['IRR'] = df.apply(lambda row: compute_irr(
    row['Investment_Cost'], row['Expected_Cashflow_Year1'],
    row['Expected_Cashflow_Year2'], row['Expected_Cashflow_Year3']), axis=1)

# ── 1D: Additional ratio features ────────────────────────────────────────────
df['Total_CF']        = df[['Expected_Cashflow_Year1',
                             'Expected_Cashflow_Year2',
                             'Expected_Cashflow_Year3']].sum(axis=1)
df['CF_to_Investment'] = (df['Total_CF'] / df['Investment_Cost']).round(3)
df['ROI_x_Growth']    = df['Historical_ROI'] * df['Market_Growth']

print('Financial features computed:')
print(df[['Project_ID','Department','Investment_Cost',
          'NPV','IRR','PI','CF_to_Investment','Success']].to_string(index=False))

Financial features computed:
 Project_ID Department  Investment_Cost        NPV     IRR     PI  CF_to_Investment  Success
          1 Operations           184654  3706072.0  470.45 21.070            26.109        0
          2  Marketing          2054354   -81821.0    6.95  0.960             1.104        1
          3    Finance          2516182  2121152.0   49.14  1.843             2.259        1
          4 Operations          3574675  -688353.0   -1.99  0.807             0.963        1
          5 Operations          2752991  -650973.0   -5.97  0.764             0.899        0
          6  Marketing          2528388  -252243.0    4.80  0.900             1.108        0
          7    Finance           628178  2054605.0  133.56  4.271             5.321        0
          8    Finance          2763046  -327587.0    4.59  0.881             1.121        1
          9 Operations          2686644  1360695.0   36.10  1.506             1.825        1
         10         IT          2670406  

In [5]:
df.head()

,Project_ID,Department,Investment_Cost,Expected_Cashflow_Year1,Expected_Cashflow_Year2,Expected_Cashflow_Year3,Historical_ROI,Market_Growth,Project_Risk,Success,NPV,PI,IRR,Total_CF,CF_to_Investment,ROI_x_Growth
0,1,Operations,184654,583556,2348816,1888756,0.08,0.05,Low,0,3706072.0,21.070,470.45,4821128,26.109,0.0040
1,2,Marketing,2054354,1447527,541234,278576,0.15,-0.01,Low,1,-81821.0,0.960,6.95,2267337,1.104,-0.0015
2,3,Finance,2516182,1372905,1998260,2312990,0.19,0.02,High,1,2121152.0,1.843,49.14,5684155,2.259,0.0038
3,4,Operations,3574675,1817640,189182,1434250,0.22,0.10,Medium,1,-688353.0,0.807,-1.99,3441072,0.963,0.0220
4,5,Operations,2752991,792139,1571101,111087,0.12,0.09,High,0,-650973.0,0.764,-5.97,2474327,0.899,0.0108


---
## Section 2 — Prepare Features for ML

In [9]:
# One-hot encode categorical columns
df_enc = pd.get_dummies(df, columns=['Department', 'Project_Risk'], drop_first=False)

# Feature set for ML models
# Note: we deliberately EXCLUDE the raw cashflow columns when predicting cashflows,
# and INCLUDE them (along with NPV/IRR) when predicting success.

BASE_FEATURES = (
    ['Investment_Cost', 'Historical_ROI', 'Market_Growth', 'ROI_x_Growth']
    + [c for c in df_enc.columns if c.startswith('Department_')]
    + [c for c in df_enc.columns if c.startswith('Project_Risk_')]
)

FULL_FEATURES = BASE_FEATURES + ['NPV', 'IRR', 'PI', 'CF_to_Investment',
                                   'Expected_Cashflow_Year1',
                                   'Expected_Cashflow_Year2',
                                   'Expected_Cashflow_Year3']

# Targets
y_cf1     = df['Expected_Cashflow_Year1'].values
y_cf2     = df['Expected_Cashflow_Year2'].values
y_cf3     = df['Expected_Cashflow_Year3'].values
y_success = df['Success'].values

X_base = df_enc[BASE_FEATURES].values.astype(float)
X_full = df_enc[FULL_FEATURES].values.astype(float)

print(f'Base features (for cash flow regression):  {len(BASE_FEATURES)}')
print(f'Full features (for success classification): {len(FULL_FEATURES)}')
print(f'Samples: {X_base.shape[0]}')

Base features (for cash flow regression):  11
Full features (for success classification): 18
Samples: 50


---
## Section 3 — Task 1: Forecast Cash Flows (Regression)

**Goal:** Given project characteristics (Department, Risk, Historical ROI, Market Growth, Investment size),  
predict what cash flows the project will generate in each year.

**Why this matters:** Department heads' projections can be optimistic. An independent ML estimate
acts as a sanity check — if their projected CF2 is ₹50M but our model predicts ₹8M, the projection is suspicious.

**Model:** Random Forest Regressor. 

**Evaluation:** 5-fold Cross-Validation → MAE and R²

In [10]:
# ── Regression: Forecast Cash Flows ───────────────────────────────────────────
#
# We predict each year's cash flow separately using the BASE features
# (project characteristics that exist BEFORE cash flows are projected)

reg_results = {}
predicted_cfs = {}

for target_name, y_target in [
    ('CF_Year1', y_cf1),
    ('CF_Year2', y_cf2),
    ('CF_Year3', y_cf3),
]:
    # ── PRIMARY: sklearn Random Forest ────────────────────────────────────
    if XGB:
        model = XGBRegressor(n_estimators=100, max_depth=4,
                                learning_rate=0.1, random_state=42,
                                verbosity=0)
    else:
        model = RandomForestRegressor(n_estimators=100, max_depth=4,
                                        random_state=42, n_jobs=-1)

    cv     = KFold(n_splits=5, shuffle=True, random_state=42)
    oof    = cross_val_predict(model, X_base, y_target, cv=cv)
    mae_cv = mean_absolute_error(y_target, oof)
    r2_cv  = r2_score(y_target, oof)

    # Fit on full data to get predictions for all projects
    model.fit(X_base, y_target)
    predicted_cfs[target_name] = model.predict(X_base)

    reg_results[target_name] = {'MAE': mae_cv, 'R2': r2_cv}

print(f"{'Target':<12} {'CV MAE':>15} {'CV R²':>8}")
print('─' * 40)
for name, res in reg_results.items():
    print(f"{name:<12} ₹{res['MAE']:>13,.0f} {res['R2']:>8.3f}")

model_label = 'XGBoost' if XGB else ('Random Forest (sklearn)' if SKLEARN else 'Random Forest (numpy)')
print(f'\nModel used: {model_label}')

Target                CV MAE    CV R²
────────────────────────────────────────
CF_Year1     ₹      590,320   -0.475
CF_Year2     ₹      778,480   -0.534
CF_Year3     ₹      863,536   -0.257

Model used: XGBoost


In [8]:
# Compute predicted NPV from predicted cash flows
df['Pred_CF1'] = predicted_cfs['CF_Year1'].round(0)
df['Pred_CF2'] = predicted_cfs['CF_Year2'].round(0)
df['Pred_CF3'] = predicted_cfs['CF_Year3'].round(0)
df['Pred_NPV'] = (
    -df['Investment_Cost']
    + df['Pred_CF1'] / (1+r)**1
    + df['Pred_CF2'] / (1+r)**2
    + df['Pred_CF3'] / (1+r)**3
).round(0)

print('Projected vs ML-estimated NPV (first 10 projects):')
comp = df[['Project_ID','Department','NPV','Pred_NPV','Success']].copy()
comp['Gap'] = (df['Pred_NPV'] - df['NPV']).round(0)
comp['Flag'] = comp['Gap'].apply(
    lambda x: '🔴 Over-projected' if x < -500000 else
              ('🟠 Under-projected' if x > 500000 else '🟢 Reasonable'))
print(comp.head(10).to_string(index=False))

Projected vs ML-estimated NPV (first 10 projects):
 Project_ID Department       NPV  Pred_NPV  Success       Gap         Flag
          1 Operations 3706072.0 3687924.0        0  -18148.0 🟢 Reasonable
          2  Marketing  -81821.0    -745.0        1   81076.0 🟢 Reasonable
          3    Finance 2121152.0 1852596.0        1 -268556.0 🟢 Reasonable
          4 Operations -688353.0 -713698.0        1  -25345.0 🟢 Reasonable
          5 Operations -650973.0 -614890.0        0   36083.0 🟢 Reasonable
          6  Marketing -252243.0 -268076.0        0  -15833.0 🟢 Reasonable
          7    Finance 2054605.0 2035829.0        0  -18776.0 🟢 Reasonable
          8    Finance -327587.0 -224235.0        1  103352.0 🟢 Reasonable
          9 Operations 1360695.0 1314646.0        1  -46049.0 🟢 Reasonable
         10         IT 1077058.0 1027134.0        1  -49924.0 🟢 Reasonable


---
## Section 4 — Task 2: Predict Project Success (Classification)

**Goal:** Predict the probability that a project will actually succeed — P(Success).  
This is the core ML task. Projects can have great NPV projections but still fail (optimism bias).

**Features:** All financial metrics + project characteristics + predicted NPV from regression above.  
**Target:** Success (0/1)  
**Model:** XGBoost Classifier (Random Forest if XGBoost unavailable)  
**Evaluation:** AUC-ROC, Precision, Recall, F1 via Stratified 5-fold CV

In [ ]:
# Add predicted NPV and cash flows as features for the classifier
df_enc2 = pd.get_dummies(df, columns=['Department', 'Project_Risk'], drop_first=False)

CLASSIFIER_FEATURES = (
    ['Investment_Cost', 'Historical_ROI', 'Market_Growth', 'ROI_x_Growth',
     'NPV', 'IRR', 'PI', 'CF_to_Investment',
     'Expected_Cashflow_Year1', 'Expected_Cashflow_Year2', 'Expected_Cashflow_Year3',
     'Pred_CF1', 'Pred_CF2', 'Pred_CF3', 'Pred_NPV']
    + [c for c in df_enc2.columns if c.startswith('Department_')]
    + [c for c in df_enc2.columns if c.startswith('Project_Risk_')]
)

X_clf = df_enc2[CLASSIFIER_FEATURES].fillna(df_enc2[CLASSIFIER_FEATURES].median()).values.astype(float)
y_clf = y_success

print(f'Classifier feature count: {len(CLASSIFIER_FEATURES)}')
print(f'Class distribution — Success=1: {y_clf.sum()}, Success=0: {(y_clf==0).sum()}')

In [ ]:
if SKLEARN:
    # ── PRIMARY: XGBoost or Random Forest classifier ───────────────────────────
    if XGB:
        clf_model = XGBClassifier(n_estimators=100, max_depth=3,
                                   learning_rate=0.1, use_label_encoder=False,
                                   eval_metric='logloss', random_state=42,
                                   scale_pos_weight=(y_clf==0).sum()/y_clf.sum())
    else:
        clf_model = RandomForestClassifier(n_estimators=100, max_depth=4,
                                            class_weight='balanced',
                                            random_state=42, n_jobs=-1)

    # Stratified CV preserves class ratio in each fold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Cross-validated predicted probabilities (out-of-fold)
    prob_oof = cross_val_predict(clf_model, X_clf, y_clf,
                                  cv=skf, method='predict_proba')[:, 1]
    pred_oof = (prob_oof >= 0.5).astype(int)

    auc = roc_auc_score(y_clf, prob_oof)
    print(f'AUC-ROC (5-fold CV): {auc:.3f}')
    print()
    print(classification_report(y_clf, pred_oof,
                                  target_names=['Failed','Succeeded']))

    # Fit on full data for final predictions
    clf_model.fit(X_clf, y_clf)
    p_success = clf_model.predict_proba(X_clf)[:, 1]

else:
    # ── FALLBACK: numpy Random Forest ─────────────────────────────────────────
    # For classification we train the tree to predict 0/1 directly
    # and use the average across trees as a pseudo-probability
    rng_cv = np.random.RandomState(42)
    idx    = rng_cv.permutation(len(y_clf))
    folds  = np.array_split(idx, 5)
    prob_oof = np.zeros(len(y_clf))

    for i in range(5):
        te = folds[i]
        tr = np.concatenate([folds[j] for j in range(5) if j != i])
        m  = _NumpyRF(n_estimators=80, max_depth=4, seed=42).fit(X_clf[tr], y_clf[tr].astype(float))
        prob_oof[te] = m.predict(X_clf[te]).clip(0, 1)

    # Fit on full data
    clf_np = _NumpyRF(n_estimators=80, max_depth=4, seed=42).fit(X_clf, y_clf.astype(float))
    p_success = clf_np.predict(X_clf).clip(0, 1)

    # Simple metrics
    pred_oof = (prob_oof >= 0.5).astype(int)
    accuracy = (pred_oof == y_clf).mean()
    # Manual AUC via sorting
    sorted_idx = np.argsort(prob_oof)
    n_pos = y_clf.sum(); n_neg = len(y_clf) - n_pos
    auc = sum((prob_oof[i] > prob_oof[j]) for i in np.where(y_clf==1)[0]
                                           for j in np.where(y_clf==0)[0]) / (n_pos * n_neg)
    print(f'CV Accuracy: {accuracy:.3f}   AUC-ROC: {auc:.3f}')

df['P_Success'] = p_success.round(3)
print(f'\nP(Success) range: {p_success.min():.2f} – {p_success.max():.2f}')

---
## Section 5 — Task 3: Expected Economic Value (EV) & Risk-Adjusted Ranking

**Expected Value** combines what the model thinks will happen (probability) with the financial outcome:

$$EV = P(\text{Success}) \times NPV_{\text{if success}} + (1 - P(\text{Success})) \times (-\text{Investment})$$

A project with 90% NPV but only 10% success probability is less valuable than one with 60% NPV and 80% success probability.  
EV captures this trade-off in a single number.

In [ ]:
# ── Expected Value ─────────────────────────────────────────────────────────────
# If project succeeds → gain is NPV
# If project fails    → loss is the full investment (sunk cost)
df['EV'] = (
    df['P_Success'] * df['NPV']
    + (1 - df['P_Success']) * (-df['Investment_Cost'])
).round(0)

# ── Risk-Adjusted Return ───────────────────────────────────────────────────────
# Penalise high-risk projects even if EV is attractive
risk_factor = df['Project_Risk'].map({'Low': 1.00, 'Medium': 0.90, 'High': 0.75})
df['Risk_Adj_EV'] = (df['EV'] * risk_factor).round(0)

# ── Final Ranking Score ────────────────────────────────────────────────────────
# Weighted combination: EV (60%) + IRR vs WACC bonus (20%) + PI (20%)
def minmax(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)

df['Score'] = (
    minmax(df['Risk_Adj_EV']) * 0.60
    + minmax(df['IRR'].fillna(0)) * 0.20
    + minmax(df['PI']) * 0.20
) * 100
df['Score'] = df['Score'].round(1)

# ── Recommendation label ────────────────────────────────────────────────────────
def recommend(row):
    if row['EV'] < 0:
        return 'REJECT'
    elif row['Score'] >= 65:
        return 'FUND'
    elif row['Score'] >= 40:
        return 'REVIEW'
    else:
        return 'HOLD'

df['Recommendation'] = df.apply(recommend, axis=1)

print('EV and ranking computed ✓')
print(df['Recommendation'].value_counts().to_string())

In [ ]:
# ── Ranked Project List (the primary deliverable) ─────────────────────────────
DISPLAY_COLS = ['Project_ID', 'Department', 'Project_Risk', 'Investment_Cost',
                'NPV', 'IRR', 'PI', 'P_Success', 'EV', 'Risk_Adj_EV',
                'Score', 'Recommendation', 'Success']

ranked = df[DISPLAY_COLS].sort_values('Score', ascending=False).reset_index(drop=True)
ranked.index += 1
ranked.index.name = 'Rank'

# Format for readability
ranked_fmt = ranked.copy()
for col in ['Investment_Cost', 'NPV', 'EV', 'Risk_Adj_EV']:
    ranked_fmt[col] = ranked_fmt[col].apply(lambda x: f'₹{x/1e6:.2f}M')
ranked_fmt['IRR']       = ranked_fmt['IRR'].apply(lambda x: f'{x:.1f}%' if pd.notna(x) else 'N/A')
ranked_fmt['P_Success'] = ranked_fmt['P_Success'].apply(lambda x: f'{x:.0%}')

print('═'*100)
print('              AURORA FINANCE — RANKED PROJECT FUNDING RECOMMENDATION')
print('═'*100)
print(ranked_fmt.to_string())
print('═'*100)

---
## Section 6 — SHAP / Feature Importance Plots

**Why SHAP?** The model predicts that Project X will succeed with 78% probability.  
An executive will ask: *why?* SHAP answers this by showing which features **pushed the prediction up or down**  
for that specific project.

If SHAP is not installed, we use feature importance as a global proxy.

In [ ]:
if SHAP and SKLEARN:
    # ── PRIMARY: SHAP plots ────────────────────────────────────────────────────
    explainer   = shap.TreeExplainer(clf_model)
    shap_values = explainer.shap_values(X_clf)

    # For Random Forest: shap_values is a list [class0, class1]
    # For XGBoost: shap_values is a single array (for binary classification)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    # Plot 1: Beeswarm — global feature importance with direction
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, X_clf, feature_names=CLASSIFIER_FEATURES,
                      plot_type='dot', show=False)
    plt.title('SHAP Summary — What drives P(Success)?',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_summary.png', bbox_inches='tight')
    plt.show()

    # Plot 2: Bar — mean absolute SHAP (magnitude of impact)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(sv, X_clf, feature_names=CLASSIFIER_FEATURES,
                      plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (Mean |SHAP value|)',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_importance.png', bbox_inches='tight')
    plt.show()

    # Plot 3: Waterfall for top project and bottom project
    top_idx    = ranked.index[0] - 1     # highest scoring project (0-based)
    bottom_idx = ranked.index[-1] - 1    # lowest scoring project (0-based)
    for idx, label in [(top_idx, 'Top Project'), (bottom_idx, 'Bottom Project')]:
        plt.figure(figsize=(10, 5))
        shap.waterfall_plot(
            shap.Explanation(values=sv[idx],
                             base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                             else explainer.expected_value,
                             data=X_clf[idx],
                             feature_names=CLASSIFIER_FEATURES),
            show=False)
        plt.title(f'SHAP Waterfall — {label} (Project {int(df["Project_ID"].iloc[idx])})',
                  fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'shap_waterfall_{label.replace(" ","_").lower()}.png', bbox_inches='tight')
        plt.show()

else:
    # ── FALLBACK: Feature Importance from model ────────────────────────────────
    print('NOTE: Install shap for full SHAP plots. Showing feature importance as proxy.\n')

    # Get importances
    if SKLEARN:
        importances = clf_model.feature_importances_
    else:
        importances = clf_np.feature_importances_

    # Keep only features with non-zero importance, top 15
    feat_imp = pd.Series(importances, index=CLASSIFIER_FEATURES).sort_values(ascending=False)
    feat_imp = feat_imp[feat_imp > 0].head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#e74c3c' if 'Risk' in f else '#3498db' if 'Dept' in f
              else '#27ae60' if 'NPV' in f or 'IRR' in f or 'PI' in f or 'EV' in f
              else '#f39c12' for f in feat_imp.index]
    feat_imp.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
    ax.invert_yaxis()
    ax.set_title('Feature Importance for P(Success) Prediction\n'
                 '(Proxy for SHAP — install shap library for full explanations)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Importance Score', fontsize=11)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='#27ae60', label='Financial metrics (NPV/IRR/PI)'),
        Patch(color='#f39c12', label='Cash flow features'),
        Patch(color='#3498db', label='Department'),
        Patch(color='#e74c3c', label='Risk level'),
    ], fontsize=9, loc='lower right')

    plt.tight_layout()
    plt.savefig('feature_importance_shap_proxy.png', bbox_inches='tight')
    plt.show()

    print('\nTop 5 features driving P(Success):')
    for feat, val in feat_imp.head(5).items():
        print(f'  {feat:<35} {val:.4f}')

---
## Section 7 — Visualisations

In [ ]:
# ── Fig 1: P(Success) vs NPV — the key quadrant chart ─────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

rec_colors = {'FUND':'#27ae60', 'REVIEW':'#f39c12',
              'HOLD':'#e67e22', 'REJECT':'#e74c3c'}
for rec, grp in df.groupby('Recommendation'):
    ax.scatter(grp['NPV']/1e6, grp['P_Success'],
               c=rec_colors[rec], label=rec, s=90,
               edgecolors='white', linewidths=0.8, zorder=3)

# Quadrant lines
npv_med  = df['NPV'].median() / 1e6
prob_med = df['P_Success'].median()
ax.axvline(npv_med,  color='grey', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(prob_med, color='grey', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)

# Quadrant labels
ax.text(df['NPV'].max()/1e6*0.6, 0.92, 'High NPV\nHigh P(Success)\n→ TOP PRIORITY',
        fontsize=8, ha='center', color='#27ae60',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
ax.text(df['NPV'].min()/1e6*0.6, 0.92, 'Low NPV\nHigh P(Success)\n→ REVIEW',
        fontsize=8, ha='center', color='#f39c12',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
ax.text(df['NPV'].max()/1e6*0.6, 0.10, 'High NPV\nLow P(Success)\n→ HIGH RISK',
        fontsize=8, ha='center', color='#e67e22',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
ax.text(df['NPV'].min()/1e6*0.6, 0.10, 'Low NPV\nLow P(Success)\n→ REJECT',
        fontsize=8, ha='center', color='#e74c3c',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Annotate top 5 projects
top5_ids = ranked.head(5)['Project_ID'].values
for _, row in df[df['Project_ID'].isin(top5_ids)].iterrows():
    ax.annotate(f"P{int(row['Project_ID'])}",
                (row['NPV']/1e6, row['P_Success']),
                textcoords='offset points', xytext=(6, 4),
                fontsize=8, fontweight='bold', color='#1a1a1a')

ax.set_xlabel('Projected NPV (₹ Millions)', fontsize=12)
ax.set_ylabel('P(Success) — ML Predicted', fontsize=12)
ax.set_title('Project Portfolio: NPV vs P(Success)\nSize = Investment Cost',
             fontsize=13, fontweight='bold')
ax.legend(title='Recommendation', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:.0f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('fig_npv_vs_psuccess.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: Top 15 projects ranked by EV ───────────────────────────────────────
top15 = ranked.head(15).copy()

fig, ax = plt.subplots(figsize=(13, 7))

bar_colors = top15['Recommendation'].map(rec_colors).values
bar_labels = [
    f"P{int(r['Project_ID'])} | {r['Department']} | "
    f"Risk: {r['Project_Risk']} | NPV: ₹{r['NPV']/1e6:.1f}M | "
    f"P(S): {r['P_Success']:.0%} | {'✓' if r['Success']==1 else '✗'}"
    for _, r in top15.iterrows()
]

bars = ax.barh(range(len(top15)), top15['Score'],
               color=bar_colors, edgecolor='white', linewidth=0.6, alpha=0.9)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(bar_labels, fontsize=8.5)
ax.invert_yaxis()
ax.set_xlabel('Composite Score (EV × Risk-Adj, 0–100)', fontsize=11)
ax.set_title('Top 15 Projects — EV-Based Ranking\n'
             'Green=FUND  Orange=REVIEW  Red=REJECT  ✓=Actually Succeeded  ✗=Actually Failed',
             fontsize=12, fontweight='bold')
ax.axvline(65, color='#27ae60', linestyle='--', linewidth=1.5, alpha=0.7, label='FUND threshold (65)')
ax.axvline(40, color='#f39c12', linestyle='--', linewidth=1.5, alpha=0.7, label='REVIEW threshold (40)')
ax.legend(fontsize=9)
ax.set_xlim(0, 110)

for bar, score in zip(bars, top15['Score']):
    ax.text(score + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.savefig('fig_top15_ranking.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: EV Distribution by Department and Risk ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Department
dept_ev = df.groupby('Department')['EV'].mean().sort_values(ascending=False) / 1e6
dept_ev.plot(kind='bar', ax=axes[0], color=sns.color_palette('Blues_d', len(dept_ev)),
             edgecolor='white')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Average EV by Department', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Avg Expected Value (₹M)', fontsize=11)
axes[0].tick_params(axis='x', rotation=15)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'₹{bar.get_height():.1f}M', ha='center', fontsize=9)

# Risk × P(Success)
risk_order = ['Low', 'Medium', 'High']
risk_pal   = {'Low':'#2ecc71','Medium':'#f39c12','High':'#e74c3c'}
for risk in risk_order:
    grp = df[df['Project_Risk']==risk]
    axes[1].scatter(grp['NPV']/1e6, grp['P_Success'],
                    color=risk_pal[risk], label=f'{risk} Risk',
                    s=80, edgecolors='white', linewidths=0.5)
axes[1].set_title('NPV vs P(Success) by Risk Level', fontsize=12, fontweight='bold')
axes[1].set_xlabel('NPV (₹M)', fontsize=11)
axes[1].set_ylabel('P(Success)', fontsize=11)
axes[1].legend()
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('fig_ev_by_dept_risk.png', bbox_inches='tight')
plt.show()

---
## Section 8 — Executive Summary

This is what you present to Aurora's Investment Committee.

In [ ]:
# ── Portfolio-level statistics ────────────────────────────────────────────────
fund_df   = df[df['Recommendation'] == 'FUND']
review_df = df[df['Recommendation'] == 'REVIEW']
reject_df = df[df['Recommendation'].isin(['REJECT', 'HOLD'])]

total_capital    = df['Investment_Cost'].sum()
fund_capital     = fund_df['Investment_Cost'].sum()
fund_total_ev    = fund_df['EV'].sum()
fund_avg_psuccess = fund_df['P_Success'].mean()

print('╔' + '═'*72 + '╗')
print('║' + '  AURORA FINANCE — INVESTMENT COMMITTEE BRIEF'.center(72) + '║')
print('║' + '  Step 1: Corporate Project Funding Recommendation'.center(72) + '║')
print('╠' + '═'*72 + '╣')
print('║' + ''.center(72) + '║')
print('║' + f'  Projects Evaluated:        {len(df):>3}'.ljust(72) + '║')
print('║' + f'  Total Capital Available:   ₹{total_capital/1e6:.1f}M'.ljust(72) + '║')
print('║' + ''.center(72) + '║')
print('╠' + '─'*72 + '╣')
print('║' + '  RECOMMENDATIONS:'.ljust(72) + '║')
print('║' + f'  ✅ FUND   → {len(fund_df):>2} projects │ Capital: ₹{fund_capital/1e6:.1f}M │ Avg P(S): {fund_avg_psuccess:.0%} │ Total EV: ₹{fund_total_ev/1e6:.1f}M'.ljust(72) + '║')
print('║' + f'  🔍 REVIEW → {len(review_df):>2} projects │ Require further due diligence'.ljust(72) + '║')
print('║' + f'  ❌ HOLD/REJECT → {len(reject_df):>2} projects │ Negative EV or insufficient return'.ljust(72) + '║')
print('╠' + '─'*72 + '╣')
print('║' + '  TOP 5 PROJECTS TO FUND:'.ljust(72) + '║')
for i, (_, row) in enumerate(ranked.head(5).iterrows(), 1):
    line = f'  {i}. P{int(row["Project_ID"])} ({row["Department"]}) │ NPV: ₹{row["NPV"]/1e6:.1f}M │ P(S): {row["P_Success"]:.0%} │ EV: ₹{row["EV"]/1e6:.1f}M'
    print('║' + line.ljust(72) + '║')
print('╠' + '─'*72 + '╣')
print('║' + '  KEY INSIGHTS:'.ljust(72) + '║')
best_dept = df.groupby('Department')['EV'].mean().idxmax()
worst_dept = df.groupby('Department')['EV'].mean().idxmin()
print('║' + f'  • {best_dept} dept generates highest avg EV — prioritise their projects'.ljust(72) + '║')
print('║' + f'  • {worst_dept} dept shows lowest avg EV — require stricter due diligence'.ljust(72) + '║')
print('║' + f'  • High-risk projects show {df[df["Project_Risk"]=="High"]["P_Success"].mean():.0%} avg P(Success) vs'.ljust(72) + '║')
print('║' + f'    {df[df["Project_Risk"]=="Low"]["P_Success"].mean():.0%} for Low-risk — risk premium is justified'.ljust(72) + '║')
print('║' + f'  • Model AUC-ROC: {auc:.2f} — reliably separates successful from failed projects'.ljust(72) + '║')
print('║' + '  • NPV projections alone are insufficient — ML adds P(Success) signal'.ljust(72) + '║')
print('╚' + '═'*72 + '╝')

In [ ]:
# ── Save final output ─────────────────────────────────────────────────────────
output_cols = ['Project_ID','Department','Project_Risk','Investment_Cost',
               'NPV','IRR','PI','P_Success','EV','Risk_Adj_EV',
               'Score','Recommendation','Success']

final_output = df[output_cols].sort_values('Score', ascending=False)
final_output.to_csv('Step1_Final_Ranked_Projects.csv', index=False)
print('Saved: Step1_Final_Ranked_Projects.csv ✓')
print('\nThis file contains the complete ranked project list — the primary submission deliverable.')